<a href="https://colab.research.google.com/github/pymood1/GPT-SoVITS-improve/blob/main/Colab-Inference-Fast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS Improve — Fast Full Install

기능은 줄이지 않고 설치/다운로드만 최적화한 테스트 노트북입니다. **Python 3.10~3.14**를 지원하며, Colab에서는 GPU 런타임을 권장합니다. Python 3.14에서는 NumPy 2.x·Gradio 5.x 등 최신 호환 의존성을 자동 선택합니다. 대용량 Hugging Face 파일은 Xet을 우선 사용하고, 진행률 redraw는 숨겨 Colab 출력 렉을 줄입니다. 감정 판단용 emotion2vec+ large와 base를 설치 단계에서 둘 다 미리 받고, large를 기본 판정 모델로 사용합니다.

앞으로 설치 로직은 `tools/colab_fast_setup.py`만 갱신하므로 이 노트북 링크는 계속 그대로 사용할 수 있습니다.


## Google Drive 마운트

설치 전에 Google Drive를 먼저 연결합니다.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
if DRIVE_ROOT.is_dir():
    print("✅ Google Drive already mounted:", DRIVE_ROOT)
else:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time


REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/content/GPT-SoVITS")


print("[1/2] GPT-SoVITS-improve 최신 main 준비", flush=True)

start = time.perf_counter()


if (REPO_DIR / ".git").exists():
    print("[git] 기존 repo 유지 + 최신 main 동기화", flush=True)

    subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )

    subprocess.run(
        ["git", "reset", "--hard", "origin/main"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    print("[git] 새 repo shallow clone", flush=True)

    clone_cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "--single-branch",
        "--branch",
        "main",
        REPO_URL,
        str(REPO_DIR),
    ]

    subprocess.run(clone_cmd, check=True)


elapsed = time.perf_counter() - start
print(f"[1/2] repo 준비 완료: {elapsed:.1f}s", flush=True)


print("[2/2] 전체 의존성 + pretrained 설치 시작", flush=True)

setup_script = REPO_DIR / "tools" / "colab_fast_setup.py"


print("[check] setup script 문법 검사", flush=True)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(setup_script)],
    check=True,
)


print("[run] setup script 실행 — 최근 로그만 표시", flush=True)

from collections import deque
from IPython.display import clear_output

setup_log = Path("/content/gpt_sovits_setup.log")
tail = deque(maxlen=24)
pending = ""
last_render = 0.0


def render_tail(final=False):
    global last_render
    now = time.monotonic()
    if not final and now - last_render < 0.8:
        return
    last_render = now
    clear_output(wait=True)
    print("[1/2] repo 준비 완료")
    print("[2/2] 전체 의존성 + pretrained 설치")
    print(f"[log] 전체 로그: {setup_log}")
    print("-" * 72)
    if tail:
        print("\n".join(tail))
    else:
        print("설치 프로세스 시작 중...")


with setup_log.open("w", encoding="utf-8") as log_writer:
    process = subprocess.Popen(
        [sys.executable, "-u", str(setup_script)],
        stdout=log_writer,
        stderr=subprocess.STDOUT,
        text=True,
    )

    with setup_log.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as log_reader:
        while process.poll() is None:
            chunk = log_reader.read()
            if chunk:
                pending += chunk.replace("\r", "\n")
                parts = pending.split("\n")
                pending = parts.pop()
                for item in parts:
                    item = item.strip()
                    if item:
                        tail.append(item)
                render_tail()
            time.sleep(0.25)

        chunk = log_reader.read()
        if chunk:
            pending += chunk.replace("\r", "\n")

if pending:
    for item in pending.split("\n"):
        item = item.strip()
        if item:
            tail.append(item)

return_code = process.wait()
render_tail(final=True)

if return_code != 0:
    error_tail = setup_log.read_text(
        encoding="utf-8",
        errors="replace",
    ).replace("\r", "\n").splitlines()[-80:]
    clear_output(wait=True)
    print("\n".join(error_tail))
    raise RuntimeError(
        f"setup script failed with exit code {return_code}. "
        f"전체 로그: {setup_log}"
    )

print("\n[OK] 설치 완료", flush=True)


## V5dev 전체 WebUI — 한국어

설치가 끝난 뒤 실행합니다. UI 언어는 **한국어(ko_KR)**, 기본 버전은 **V5dev**로 시작합니다.

일반 GPT-SoVITS 흐름처럼 **1A-전처리 → 1B-학습 → 1C-추론** 순서로 사용합니다. 감정 데이터 준비와 controller 학습은 기존 단계 안의 선택 기능이며, 추론에서는 `[happy]`, `[sad:0.5]`, `[neutral]` 같은 감정 태그를 사용할 수 있습니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


REPO_DIR = Path("/content/GPT-SoVITS")


env = os.environ.copy()

python_path = [
    str(REPO_DIR),
    str(REPO_DIR / "GPT_SoVITS"),
]

existing_python_path = env.get("PYTHONPATH")

if existing_python_path:
    python_path.append(existing_python_path)


env.update(
    {
        "PYTHONPATH": os.pathsep.join(python_path),
        "language": "ko_KR",
        "version": "v5dev",
        "is_share": "True",
        "HF_HOME": "/content/huggingface",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "HF_HUB_DISABLE_UPDATE_CHECK": "1",
        "HF_HUB_DOWNLOAD_TIMEOUT": "120",
        "HF_XET_HIGH_PERFORMANCE": "1",
    }
)


webui_cmd = [
    sys.executable,
    "-u",
    "webui.py",
    "ko_KR",
]


print("[run] 파인튜닝 WebUI 시작 — 한국어", flush=True)

process = subprocess.Popen(
    webui_cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"webui.py failed with exit code {return_code}. "
        "위에 표시된 마지막 traceback/에러 메시지를 확인하세요."
    )
